# Entrainement - Prediction de panne depuis CSV

Objectif : entrainer un premier modele qui predit si une panne va arriver bientot a partir de la telemetrie de simulation.

Le notebook part de deux fichiers :
- `dataset_ml.csv` : une ligne par `time + server_id`, avec les colonnes capteurs.
- `fan_map.csv` : mapping `fan_id -> server_id`, utile pour rattacher les evenements `CRASH_FAN` au serveur concerne.

Par defaut, le label vaut 1 si un incident arrive dans les 4 prochains ticks : `CRASH_FAN`, `THERMAL_DRIFT_SERVER` ou `LOAD_SPIKE_ALL`.

## 0. Exporter les donnees depuis TimescaleDB

A lancer depuis la racine du projet si les CSV n'existent pas encore.

```bash
docker compose -f docker-compose.yaml exec -T timescaledb \
  psql -U tsuser -d tsdb \
  -c "\COPY (
    SELECT
      sd.time,
      s.server_id,
      srv.hostname,
      MAX(CASE WHEN s.sensor_type = 'CPU_TEMP' THEN sd.value END) AS cpu_temp,
      MAX(CASE WHEN s.sensor_type = 'LOAD' THEN sd.value END) AS cpu_load,
      MAX(CASE WHEN s.sensor_type = 'TOTAL_POWER' THEN sd.value END) AS total_power,
      MAX(CASE WHEN s.sensor_type = 'FAN_SPEED_1' THEN sd.value END) AS fan_speed_1,
      MAX(CASE WHEN s.sensor_type = 'FAN_SPEED_2' THEN sd.value END) AS fan_speed_2
    FROM sensor_data sd
    JOIN sensor s ON s.sensor_id = sd.sensor_id
    JOIN server srv ON srv.server_id = s.server_id
    GROUP BY sd.time, s.server_id, srv.hostname
    ORDER BY sd.time, s.server_id
  ) TO STDOUT WITH CSV HEADER" > dataset_ml.csv

docker compose -f docker-compose.yaml exec -T timescaledb \
  psql -U tsuser -d tsdb \
  -c "\COPY (
    SELECT fan_id, server_id
    FROM fan
    ORDER BY fan_id
  ) TO STDOUT WITH CSV HEADER" > fan_map.csv
```

`*.csv` est ignore par Git dans ce projet, donc ces fichiers restent locaux.

In [ ]:
# Si besoin, decommenter cette cellule
# %pip install pandas numpy matplotlib scikit-learn joblib

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path("..").resolve()
DATASET_PATH = PROJECT_ROOT / "dataset_ml.csv"
FAN_MAP_PATH = PROJECT_ROOT / "fan_map.csv"
SCENARIO_PATH = PROJECT_ROOT / "nodejs-server" / "src" / "data_seed" / "scenarios.json"
MODEL_PATH = Path("failure_prediction_model.joblib")

PREDICTED_EVENT_TYPES = ["CRASH_FAN", "THERMAL_DRIFT_SERVER", "LOAD_SPIKE_ALL"]
HORIZON_TICKS = 4

assert DATASET_PATH.exists(), f"Dataset introuvable: {DATASET_PATH}"
assert SCENARIO_PATH.exists(), f"Scenario introuvable: {SCENARIO_PATH}"

DATASET_PATH, FAN_MAP_PATH, SCENARIO_PATH

## 1. Charger et nettoyer la telemetrie

In [ ]:
df = pd.read_csv(DATASET_PATH, parse_dates=["time"])
df.columns = [c.strip().lower() for c in df.columns]

expected = {"time", "server_id", "hostname", "cpu_temp", "cpu_load", "total_power", "fan_speed_1", "fan_speed_2"}
missing = expected - set(df.columns)
assert not missing, f"Colonnes manquantes: {missing}"

df = df.sort_values(["server_id", "time"]).copy()

# Si un timestamp a ete insere deux fois, on garde une seule ligne agregee.
df = (
    df.groupby(["time", "server_id", "hostname"], as_index=False)
      .agg({
          "cpu_temp": "mean",
          "cpu_load": "mean",
          "total_power": "mean",
          "fan_speed_1": "mean",
          "fan_speed_2": "mean",
      })
      .sort_values(["server_id", "time"])
      .reset_index(drop=True)
)

print(df.shape)
display(df.head())
display(df.tail())

In [ ]:
summary = df.agg({"time": ["min", "max"], "server_id": "nunique"})
display(summary)

rows_per_sensor_time = df.groupby("time").size().describe()
display(rows_per_sensor_time)

## 2. Charger le scenario et construire les labels

`THERMAL_DRIFT_SERVER` cible directement un `server_id`.

`CRASH_FAN` cible un `fan_id`, donc on utilise `fan_map.csv` pour retrouver le serveur concerne. Si `fan_map.csv` n'est pas present, le notebook ne pourra pas creer des labels propres pour `CRASH_FAN`.

`LOAD_SPIKE_ALL` est global : il concerne tous les serveurs.

In [ ]:
with open(SCENARIO_PATH, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

scenario = next(s for s in scenarios if s["id"] == "sc_marseille_gpu_melt")
events = pd.DataFrame(scenario["events"])
events = events[events["type"].isin(PREDICTED_EVENT_TYPES)].copy()

fan_map = pd.DataFrame(columns=["fan_id", "server_id"])
if FAN_MAP_PATH.exists():
    fan_map = pd.read_csv(FAN_MAP_PATH)
    fan_map.columns = [c.strip().lower() for c in fan_map.columns]
else:
    print("ATTENTION: fan_map.csv absent. Les labels CRASH_FAN seront ignores.")

display(events.head(10))
display(fan_map.head())

In [ ]:
start_time = df["time"].min()

# On estime la duree d'un tick depuis les donnees. Dans ta simulation actuelle, c'est 1h.
time_steps = (
    df[["time"]]
    .drop_duplicates()
    .sort_values("time")
    ["time"]
    .diff()
    .dropna()
)
tick_delta = time_steps.mode().iloc[0]
print("start_time =", start_time)
print("tick_delta =", tick_delta)

all_server_ids = sorted(df["server_id"].unique())

def event_server_ids(row):
    if row["type"] == "LOAD_SPIKE_ALL":
        return all_server_ids
    if row["type"] == "THERMAL_DRIFT_SERVER":
        return [int(row["targetId"])]
    if row["type"] == "CRASH_FAN" and not fan_map.empty:
        match = fan_map.loc[fan_map["fan_id"] == int(row["targetId"]), "server_id"]
        if len(match):
            return [int(match.iloc[0])]
    return []

events["server_id"] = events.apply(event_server_ids, axis=1)
events = events.explode("server_id").dropna(subset=["server_id"]).copy()
events["server_id"] = events["server_id"].astype(int)
events["event_time"] = start_time + events["tick"].astype(int) * tick_delta

display(events[["tick", "event_time", "type", "targetId", "server_id", "value"]].head(20))

In [ ]:
df["failure_next_horizon"] = 0
df["failure_type_next"] = "none"

horizon_delta = HORIZON_TICKS * tick_delta

for event in events.itertuples(index=False):
    start_window = event.event_time - horizon_delta
    end_window = event.event_time
    mask = (
        (df["server_id"] == event.server_id)
        & (df["time"] >= start_window)
        & (df["time"] <= end_window)
    )
    df.loc[mask, "failure_next_horizon"] = 1
    df.loc[mask, "failure_type_next"] = event.type

display(df["failure_next_horizon"].value_counts(dropna=False))
display(df["failure_type_next"].value_counts(dropna=False))

## 3. Creer les features temporelles

Le modele doit voir les tendances, pas seulement la valeur instantanee.

In [ ]:
feature_df = df.copy()
sensor_cols = ["cpu_temp", "cpu_load", "total_power", "fan_speed_1", "fan_speed_2"]

for col in sensor_cols:
    group = feature_df.groupby("server_id")[col]
    feature_df[f"{col}_lag_1"] = group.shift(1)
    feature_df[f"{col}_delta_1"] = group.diff(1)
    feature_df[f"{col}_delta_3"] = group.diff(3)
    feature_df[f"{col}_mean_3"] = group.transform(lambda s: s.rolling(3, min_periods=1).mean())
    feature_df[f"{col}_mean_6"] = group.transform(lambda s: s.rolling(6, min_periods=1).mean())

feature_df["fan_speed_mean"] = feature_df[["fan_speed_1", "fan_speed_2"]].mean(axis=1)
feature_df["temp_minus_fan"] = feature_df["cpu_temp"] - feature_df["fan_speed_mean"]
feature_df["power_per_load"] = feature_df["total_power"] / feature_df["cpu_load"].replace(0, np.nan)
feature_df["hour"] = feature_df["time"].dt.hour
feature_df["dayofweek"] = feature_df["time"].dt.dayofweek

feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.dropna().reset_index(drop=True)

print(feature_df.shape)
display(feature_df.head())

## 4. Train/test split temporel

On coupe par date pour eviter de tester sur le passe d'une serie deja vue aleatoirement.

In [ ]:
target_col = "failure_next_horizon"
drop_cols = ["time", "hostname", "failure_next_horizon", "failure_type_next"]
X_cols = [c for c in feature_df.columns if c not in drop_cols]

split_time = feature_df["time"].quantile(0.75)
train_df = feature_df[feature_df["time"] <= split_time].copy()
test_df = feature_df[feature_df["time"] > split_time].copy()

X_train, y_train = train_df[X_cols], train_df[target_col]
X_test, y_test = test_df[X_cols], test_df[target_col]

print("split_time:", split_time)
print("train:", X_train.shape, y_train.value_counts(normalize=True).to_dict())
print("test :", X_test.shape, y_test.value_counts(normalize=True).to_dict())

## 5. Entrainer plusieurs baselines

In [ ]:
models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced")
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
        ap = average_precision_score(y_test, proba)
    else:
        ap = np.nan
    report = classification_report(y_test, pred, output_dict=True, zero_division=0)
    results.append({
        "model": name,
        "precision_failure": report.get("1", {}).get("precision", 0),
        "recall_failure": report.get("1", {}).get("recall", 0),
        "f1_failure": report.get("1", {}).get("f1-score", 0),
        "average_precision": ap,
        "accuracy": report.get("accuracy", 0),
    })

results_df = pd.DataFrame(results).sort_values("f1_failure", ascending=False)
display(results_df)

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = trained[best_name]
pred = best_model.predict(X_test)

print("Best model:", best_name)
print(classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title(f"Confusion matrix - {best_name}")
plt.show()

## 6. Inspecter les features importantes

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X_cols).sort_values(ascending=False)
    display(importances.head(20))
    importances.head(20).sort_values().plot(kind="barh", figsize=(8, 6))
    plt.title("Top features")
    plt.show()
else:
    print("Le meilleur modele ne fournit pas feature_importances_.")

## 7. Sauvegarder le modele

On sauvegarde aussi la liste des colonnes, car elle doit etre identique au moment de predire.

In [ ]:
bundle = {
    "model": best_model,
    "feature_columns": X_cols,
    "horizon_ticks": HORIZON_TICKS,
    "predicted_event_types": PREDICTED_EVENT_TYPES,
    "tick_delta_seconds": tick_delta.total_seconds(),
}

joblib.dump(bundle, MODEL_PATH)
MODEL_PATH.resolve()

## 8. Exemple de prediction sur les dernieres lignes

In [ ]:
sample = feature_df.sort_values("time").tail(20).copy()
sample_pred = best_model.predict(sample[X_cols])
sample_proba = best_model.predict_proba(sample[X_cols])[:, 1] if hasattr(best_model, "predict_proba") else sample_pred

out = sample[["time", "server_id", "hostname", "cpu_temp", "cpu_load", "total_power", "fan_speed_mean", "failure_next_horizon"]].copy()
out["pred_failure_next_horizon"] = sample_pred
out["pred_probability"] = sample_proba
display(out)